In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"D:\Foundathon3.0\Mess-Food-Analytics\Final_Data\final_training_data.csv")

df.head()


,Mess,Month,Meal,Waste_KG,Plates,Waste_Percent,Estimated_Dish
0,D-MESS,Apr-23,BF,1006.0,24455,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...
1,D-MESS,Apr-23,LUNCH,3161.0,31425,10,Rice/Dal/Sabzi/Curd/Palak Panneer/Chicken Curr...
2,D-MESS,Apr-23,SNACKS,0.0,31350,0,"Tea/Biscuits/Samosa/Pav Baji,Pani Poori/Bonda"
3,D-MESS,Apr-23,DINNER,3175.0,31360,10,Roti/Paneer/Chicken/Rice/Jeera Dal/Panjabi Par...
4,D-MESS,May-23,BF,1023.0,24240,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...


In [3]:
df = df[df['Plates'] > 0].reset_index(drop=True)

In [4]:
dish_weights = {
    'biscuits'        : 0.80,
    'bonda'           : 0.60,
    'bread'           : 0.90,
    'chappathi'       : 0.50,
    'chicken'         : 0.70,
    'chicken curry'   : 0.70,
    'curd'            : 0.70,
    'dal'             : 0.80,
    'dal tadka'       : 0.80,
    'dosa'            : 0.60,
    'idli'            : 0.80,
    'jeera dal'       : 0.70,
    'kashmiri dum aloo': 0.70,
    'palak panneer'   : 0.80,
    'paneer'          : 0.70,
    'panjabi paratha' : 0.50,
    'pav baji'        : 0.80,
    'pani poori'      : 0.90,
    'poha'            : 0.90,
    'pongal'          : 0.85,
    'poori'           : 0.70,
    'potato masala'   : 0.67,
    'rice'            : 0.77,
    'roti'            : 0.67,
    'sabzi'           : 0.70,
    'samosa'          : 0.80,
    'steamed rice'    : 0.90,
    'tea'             : 0.80,
    'upma'            : 0.87,
    'vada'            : 0.80,
    'veg kuruma'      : 0.67,
    'vegetable curry' : 0.66,
}

In [5]:
def parse_dishes(dish_str):
    """Split on / first, then handle 'Pav Baji,Pani Poori' as two dishes."""
    if not isinstance(dish_str, str):
        return []
    dishes = []
    for part in dish_str.split('/'):
        # handle comma-separated within a slash segment
        sub = [d.strip().lower().rstrip(',') for d in part.split(',')]
        dishes.extend([d for d in sub if d])
    return dishes

df['dish_list'] = df['Estimated_Dish'].apply(parse_dishes)

# Verify pav baji and pani poori are split correctly
sample = df[df['Meal']=='SNACKS']['dish_list'].iloc[0]
print(f"Snacks dishes parsed: {sample}")

Snacks dishes parsed: ['tea', 'biscuits', 'samosa', 'pav baji', 'pani poori', 'bonda']


In [6]:
def menu_scores(dish_list):
    scores = [dish_weights.get(d, 0.5) for d in dish_list]
    if not scores:
        return 0.5, 0.5, 0.5, 0
    return (
        np.mean(scores),   # avg popularity of the meal
        np.max(scores),    # best dish in the meal (star dish)
        np.sum(scores),    # total appeal (more dishes = more draw)
        len(scores)        # number of dishes
    )

In [7]:
df[['menu_avg', 'menu_max', 'menu_sum', 'dish_count']] = pd.DataFrame(
    df['dish_list'].apply(menu_scores).tolist(), index=df.index
)

print("\nMenu scores per meal type:")
print(df.groupby('Meal')[['menu_avg','menu_max','menu_sum']].mean().round(3).to_string())


Menu scores per meal type:
        menu_avg  menu_max  menu_sum
Meal                                
BF         0.759       0.9      7.59
DINNER     0.697       0.9      6.27
LUNCH      0.746       0.8      5.97
SNACKS     0.783       0.9      4.70


Feature Engineering

In [8]:
df['month_parsed']  = pd.to_datetime(df['Month'], format='%b-%y')
df['month_num']     = df['month_parsed'].dt.month

In [9]:
def month_to_academic_phase(m):
    if m in [6, 1]:    return 0   # Semester break
    elif m in [7]:     return 1   # Transition
    elif m in [5, 12]: return 2   # End of semester
    else:              return 3   # Full semester

df['academic_phase'] = df['month_num'].apply(month_to_academic_phase)
df['month_sin']      = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos']      = np.cos(2 * np.pi * df['month_num'] / 12)

In [10]:
mess_dummies      = pd.get_dummies(df['Mess'], prefix='mess')
meal_dummies      = pd.get_dummies(df['Meal'], prefix='meal')
df['mess_meal']   = df['Mess'] + '_' + df['Meal']
mess_meal_dummies = pd.get_dummies(df['mess_meal'], prefix='grp')

Model Training

In [11]:
X = pd.concat([
    df[['academic_phase', 'month_sin', 'month_cos', 'month_num',
        'menu_avg', 'menu_max', 'menu_sum', 'dish_count']],  # ← dish scores
    mess_dummies,
    meal_dummies,
    mess_meal_dummies
], axis=1)

y = df['Plates']

In [12]:
print(f"\nFeature matrix: {X.shape[1]} features")
print(f"   → 4 time/academic features")
print(f"   → 4 dish popularity features  ← NEW")
print(f"   → {len(mess_dummies.columns)} mess + {len(meal_dummies.columns)} meal + {len(mess_meal_dummies.columns)} interaction dummies")


Feature matrix: 42 features
   → 4 time/academic features
   → 4 dish popularity features  ← NEW
   → 6 mess + 4 meal + 24 interaction dummies


In [13]:

print("MODEL EVALUATION (5-Fold CV)")

models = {
    'Random Forest'    : RandomForestRegressor(n_estimators=300, min_samples_leaf=1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
}

MODEL EVALUATION (5-Fold CV)


In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    mae = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    r2  =  cross_val_score(model, X, y, cv=kf, scoring='r2')

    mape_scores = []
    for train_idx, val_idx in kf.split(X):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[val_idx])
        mape  = np.mean(np.abs((y.iloc[val_idx] - preds) / y.iloc[val_idx])) * 100
        mape_scores.append(mape)

    results[name] = {'MAE': mae.mean(), 'R2': r2.mean(), 'MAPE': np.mean(mape_scores)}
    print(f"\n{name}")
    print(f"  MAE  : {mae.mean():>10,.0f} ± {mae.std():,.0f} plates")
    print(f"  MAPE : {np.mean(mape_scores):>9.1f}%")
    print(f"  R²   : {r2.mean():>10.3f}")


Random Forest
  MAE  :      4,402 ± 494 plates
  MAPE :      34.2%
  R²   :      0.968

Gradient Boosting
  MAE  :      3,683 ± 494 plates
  MAPE :      32.6%
  R²   :      0.981


Train best and feature importance 

In [15]:
best_name  = min(results, key=lambda k: results[k]['MAE'])
best_model = models[best_name]
best_model.fit(X, y)
print(f"\n Best model: {best_name}")

importances = pd.Series(best_model.feature_importances_, index=X.columns)
top = importances.sort_values(ascending=False).head(15)
print("\n" + "="*55)
print("TOP 15 FEATURES")
print("="*55)
for feat, imp in top.items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<35} {imp:.4f}  {bar}")


 Best model: Gradient Boosting

TOP 15 FEATURES
  mess_SANNASI                        0.5902  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  academic_phase                      0.1457  █████████████████████████████
  month_num                           0.0432  ████████
  meal_BF                             0.0409  ████████
  menu_sum                            0.0285  █████
  grp_SANNASI_BF                      0.0249  ████
  mess_MEENAKSHI                      0.0244  ████
  month_cos                           0.0243  ████
  dish_count                          0.0239  ████
  month_sin                           0.0199  ███
  mess_STAFF MESS                     0.0169  ███
  mess_J-MESS                         0.0035  
  mess_D-MESS                         0.0034  
  grp_STAFF MESS_SNACKS               0.0016  
  mess_SENBAGAM                       0.0016  


In [16]:
df['Predicted'] = best_model.predict(X).round().astype(int)
df['Error_Pct'] = ((df['Predicted'] - df['Plates']) / df['Plates'] * 100).round(1)

print("\n" + "="*55)
print("ACTUAL vs PREDICTED")
print("="*55)
print(df[['Mess','Month','Meal','Plates','Predicted','Error_Pct']].head(24).to_string(index=False))
print(f"\nOverall avg error: {df['Error_Pct'].abs().mean():.1f}%")


ACTUAL vs PREDICTED
  Mess  Month   Meal  Plates  Predicted  Error_Pct
D-MESS Apr-23     BF   24455      22510       -8.0
D-MESS Apr-23  LUNCH   31425      31704        0.9
D-MESS Apr-23 SNACKS   31350      34032        8.6
D-MESS Apr-23 DINNER   31360      33960        8.3
D-MESS May-23     BF   24240      20034      -17.4
D-MESS May-23  LUNCH   29827      28045       -6.0
D-MESS May-23 SNACKS   32415      30208       -6.8
D-MESS May-23 DINNER   32415      30282       -6.6
D-MESS Jun-23     BF    1360       -788     -157.9
D-MESS Jun-23  LUNCH    1655       2080       25.7
D-MESS Jun-23 SNACKS     840       1656       97.1
D-MESS Jun-23 DINNER     840       1888      124.8
D-MESS Jul-23     BF   12465      14356       15.2
D-MESS Jul-23  LUNCH   19740      19577       -0.8
D-MESS Jul-23 SNACKS   21090      20389       -3.3
D-MESS Jul-23 DINNER   21090      20170       -4.4
D-MESS Aug-23     BF   24580      26334        7.1
D-MESS Aug-23  LUNCH   40135      39166       -2.4
D-MESS Aug

Predictions 

In [17]:
def predict_plates(mess, meal, month_str):
    dt = pd.to_datetime(month_str, format='%b-%y')
    m  = dt.month

    # get dish scores for this meal from training data
    meal_row = df[df['Meal'] == meal].iloc[0]
    m_avg, m_max, m_sum, d_count = (
        meal_row['menu_avg'], meal_row['menu_max'],
        meal_row['menu_sum'], meal_row['dish_count']
    )

    row = pd.DataFrame([[0]*len(X.columns)], columns=X.columns)
    row['academic_phase'] = month_to_academic_phase(m)
    row['month_sin']      = np.sin(2 * np.pi * m / 12)
    row['month_cos']      = np.cos(2 * np.pi * m / 12)
    row['month_num']      = m
    row['menu_avg']       = m_avg
    row['menu_max']       = m_max
    row['menu_sum']       = m_sum
    row['dish_count']     = d_count

    for col in [f'mess_{mess}', f'meal_{meal}', f'grp_{mess}_{meal}']:
        if col in row.columns:
            row[col] = 1

    return max(0, int(round(best_model.predict(row)[0])))

In [19]:

print("SAMPLE PREDICTIONS")

tests = [
    ('D-MESS',   'LUNCH',  'Aug-24'),
    ('SANNASI',  'DINNER', 'Jun-24'),   # break month — expect low
    ('SANNASI',  'DINNER', 'Oct-24'),   # full semester — expect high
    ('MEENAKSHI','BF',     'Feb-24'),
    ('D-MESS',   'SNACKS', 'Sep-24'),
]
for mess, meal, month in tests:
    pred = predict_plates(mess, meal, month)
    avg  = df[(df['Mess']==mess) & (df['Meal']==meal)]['Plates'].mean()
    print(f"  {mess:<12} {meal:<7} {month}  → {pred:>8,} plates  (historical avg: {avg:,.0f})")



SAMPLE PREDICTIONS
  D-MESS       LUNCH   Aug-24  →   39,166 plates  (historical avg: 25,818)
  SANNASI      DINNER  Jun-24  →   28,116 plates  (historical avg: 121,029)
  SANNASI      DINNER  Oct-24  →  157,919 plates  (historical avg: 121,029)
  MEENAKSHI    BF      Feb-24  →    7,800 plates  (historical avg: 6,675)
  D-MESS       SNACKS  Sep-24  →   40,922 plates  (historical avg: 28,118)
